# 06 — Heavy Preprocessing

## Objective

Prepare the raw e-commerce datasets for further statistical analysis and machine learning while preserving data integrity.

This notebook will investigate missing values, outliers, feature engineering, categorical encoding, and numerical scaling. Every preprocessing decision will be validated and documented before it is applied.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Directory exists:", RAW_DATA_DIR.exists())

print("\nFiles available in data/raw:")

for file_path in sorted(RAW_DATA_DIR.iterdir()):
    if file_path.is_file():
        print(f"{file_path.name} ({file_path.stat().st_size:,} bytes)")

Project root: c:\Users\Lenovo\ecommerce-customer-behaviour
Raw data directory: c:\Users\Lenovo\ecommerce-customer-behaviour\data\raw
Directory exists: True

Files available in data/raw:
customers.csv (200,841 bytes)
customers.json (438,953 bytes)
manifest.json (2,210 bytes)
order_items.csv (705,875 bytes)
order_items.json (1,899,518 bytes)
orders.csv (1,008,896 bytes)
orders.json (2,434,131 bytes)
products.csv (598,343 bytes)
products.json (1,356,901 bytes)
reviews.csv (488,124 bytes)
reviews.json (997,446 bytes)


In [2]:
# Load original CSV datasets
data = {
    "products": pd.read_csv(RAW_DATA_DIR / "products.csv"),
    "customers": pd.read_csv(RAW_DATA_DIR / "customers.csv"),
    "orders": pd.read_csv(RAW_DATA_DIR / "orders.csv"),
    "order_items": pd.read_csv(RAW_DATA_DIR / "order_items.csv"),
    "reviews": pd.read_csv(RAW_DATA_DIR / "reviews.csv"),
}

# Record original dataset dimensions
baseline_shapes = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "duplicate_rows": int(df.duplicated().sum()),
            "missing_cells": int(df.isna().sum().sum()),
        }
        for name, df in data.items()
    ]
)

display(baseline_shapes)

# Inspect column names and data types
for name, df in data.items():
    print(f"\n{'=' * 60}")
    print(f"DATASET: {name.upper()}")
    print(f"{'=' * 60}")

    print("Shape:", df.shape)
    print("\nData types:")
    display(df.dtypes.astype(str).rename("dtype").to_frame())

    print("\nMissing values by column:")
    missing = df.isna().sum()
    display(
        missing[missing > 0]
        .sort_values(ascending=False)
        .rename("missing_count")
        .to_frame()
    )

,dataset,rows,columns,duplicate_rows,missing_cells
0,products,2000,19,0,2204
1,customers,800,13,0,1280
2,orders,3000,21,0,3668
3,order_items,6219,10,0,0
4,reviews,2500,10,0,0



DATASET: PRODUCTS
Shape: (2000, 19)

Data types:


,dtype
id,int64
created_at,str
updated_at,str
sku,str
name,str
slug,str
description,str
category_id,int64
brand,str
price,float64



Missing values by column:


,missing_count
compare_at_price,1645
rating,559


...

,missing_count



DATASET: REVIEWS
Shape: (2500, 10)

Data types:


,dtype
id,int64
created_at,str
updated_at,str
product_id,int64
customer_id,int64
rating,int64
title,str
body,str
verified_purchase,bool
helpful_votes,int64



Missing values by column:


,missing_count


In [5]:
# Get the orders dataset
orders = data["orders"]

# Investigate missing order fields by status
missing_by_status = (
    orders.groupby("status")
    .agg(
        order_count=("id", "count"),
        missing_paid_at=("paid_at", lambda s: s.isna().sum()),
        missing_shipped_at=("shipped_at", lambda s: s.isna().sum()),
        missing_delivered_at=("delivered_at", lambda s: s.isna().sum()),
        missing_tracking_number=(
            "tracking_number",
            lambda s: s.isna().sum()
        ),
    )
    .reset_index()
)

display(missing_by_status)

print("\nOrders with missing paid_at:")
display(
    orders.loc[
        orders["paid_at"].isna(),
        ["id", "status", "placed_at", "paid_at",
         "shipped_at", "delivered_at"]
    ].head(20)
)

print("\nOrders with missing delivered_at:")
display(
    orders.loc[
        orders["delivered_at"].isna(),
        ["id", "status", "placed_at", "paid_at",
         "shipped_at", "delivered_at", "tracking_number"]
    ].head(20)
)

,status,order_count,missing_paid_at,missing_shipped_at,missing_delivered_at,missing_tracking_number
0,cancelled,173,0,173,173,173
1,delivered,2675,0,0,0,0
2,paid,9,0,9,9,9
3,pending,7,7,7,7,7
4,refunded,104,0,104,104,104
5,shipped,32,0,0,32,0



Orders with missing paid_at:


,id,status,placed_at,paid_at,shipped_at,delivered_at
0,1889,pending,2026-09-01T11:06:02Z,NaN,NaN,NaN
2,2283,pending,2026-09-01T09:04:56Z,NaN,NaN,NaN
3,1840,pending,2026-09-01T01:39:07Z,NaN,NaN,NaN
4,609,pending,2026-09-01T00:41:24Z,NaN,NaN,NaN
5,1314,pending,2026-08-31T16:13:54Z,NaN,NaN,NaN
6,1477,pending,2026-08-31T15:23:09Z,NaN,NaN,NaN
7,290,pending,2026-08-31T14:16:42Z,NaN,NaN,NaN



Orders with missing delivered_at:


,id,status,placed_at,paid_at,shipped_at,delivered_at,tracking_number
0,1889,pending,2026-09-01T11:06:02Z,NaN,NaN,NaN,NaN
1,1601,paid,2026-09-01T09:42:53Z,2026-09-01T10:05:53Z,NaN,NaN,NaN
2,2283,pending,2026-09-01T09:04:56Z,NaN,NaN,NaN,NaN
3,1840,pending,2026-09-01T01:39:07Z,NaN,NaN,NaN,NaN
4,609,pending,2026-09-01T00:41:24Z,NaN,NaN,NaN,NaN
5,1314,pending,2026-08-31T16:13:54Z,NaN,NaN,NaN,NaN
6,1477,pending,2026-08-31T15:23:09Z,NaN,NaN,NaN,NaN
7,290,pending,2026-08-31T14:16:42Z,NaN,NaN,NaN,NaN
8,896,cancelled,2026-08-31T06:13:58Z,2026-08-31T07:28:58Z,NaN,NaN,NaN
9,2711,shipped,2026-08-31T05:57:38Z,2026-08-31T06:19:38Z,2026-08-31T15:21:38Z,NaN,1ZCK0ZHAXFPOFPRRRR


### Missing Order Timestamps and Tracking Data

Missing order fields were investigated by order status.

- All 7 pending orders had missing payment, shipment, and delivery timestamps, consistent with their recorded status.
- All 9 paid orders lacked shipment and delivery timestamps.
- All 32 shipped orders had tracking numbers and shipment timestamps but no delivery timestamps.
- Cancelled orders lacked shipment and delivery timestamps and tracking numbers.
- Refunded orders also lacked shipment and delivery information; however, the exact meaning and lifecycle of the refunded status remain unconfirmed.
- All delivered orders had complete payment, shipment, delivery, and tracking fields.

These patterns are internally consistent with the recorded statuses, although the source system's business rules have not been independently verified.

**Preprocessing decision:** Preserve the original missing values in timestamp and tracking fields. Do not impute event timestamps or replace missing tracking numbers with fabricated values. Missingness may represent an event that has not occurred or information that was not recorded.

In [6]:
# Create separate working copies
processed = {
    name: df.copy(deep=True)
    for name, df in data.items()
}

# Convert timestamp columns to datetime
timestamp_columns = {
    "products": ["created_at", "updated_at"],
    "customers": ["created_at", "updated_at"],
    "orders": [
        "created_at",
        "updated_at",
        "placed_at",
        "paid_at",
        "shipped_at",
        "delivered_at",
    ],
    "order_items": ["created_at", "updated_at"],
    "reviews": ["created_at", "updated_at"],
}

for dataset_name, columns in timestamp_columns.items():
    for column in columns:
        processed[dataset_name][column] = pd.to_datetime(
            processed[dataset_name][column],
            utc=True,
            errors="coerce",
        )

# Validate conversion results
for dataset_name, columns in timestamp_columns.items():
    print(f"\n{dataset_name.upper()}")

    for column in columns:
        series = processed[dataset_name][column]

        print(
            f"{column}: dtype={series.dtype}, "
            f"missing={series.isna().sum()}"
        )


PRODUCTS
created_at: dtype=datetime64[us, UTC], missing=0
updated_at: dtype=datetime64[us, UTC], missing=0

CUSTOMERS
created_at: dtype=datetime64[us, UTC], missing=0
updated_at: dtype=datetime64[us, UTC], missing=0

ORDERS
created_at: dtype=datetime64[us, UTC], missing=0
updated_at: dtype=datetime64[us, UTC], missing=0
placed_at: dtype=datetime64[us, UTC], missing=0
paid_at: dtype=datetime64[us, UTC], missing=7
shipped_at: dtype=datetime64[us, UTC], missing=293
delivered_at: dtype=datetime64[us, UTC], missing=325

ORDER_ITEMS
created_at: dtype=datetime64[us, UTC], missing=0
updated_at: dtype=datetime64[us, UTC], missing=0

REVIEWS
created_at: dtype=datetime64[us, UTC], missing=0
updated_at: dtype=datetime64[us, UTC], missing=0


In [7]:
# Select important numerical columns
numeric_columns = {
    "products": ["price", "compare_at_price", "weight_g", "rating"],
    "customers": ["orders_count", "total_spent"],
    "orders": ["subtotal", "shipping", "tax", "discount", "total"],
    "order_items": ["quantity", "unit_price", "line_total"],
    "reviews": ["rating", "helpful_votes"],
}

# Generate descriptive statistics and IQR outlier counts
numeric_summary = []

for dataset_name, columns in numeric_columns.items():
    df = processed[dataset_name]

    for column in columns:
        series = df[column].dropna()

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        numeric_summary.append({
            "dataset": dataset_name,
            "column": column,
            "non_missing_count": len(series),
            "minimum": series.min(),
            "median": series.median(),
            "maximum": series.max(),
            "IQR_lower_bound": lower_bound,
            "IQR_upper_bound": upper_bound,
            "IQR_outlier_count": int(
                ((series < lower_bound) | (series > upper_bound)).sum()
            ),
        })

numeric_summary = pd.DataFrame(numeric_summary)

display(
    numeric_summary.round(2)
)

,dataset,column,non_missing_count,minimum,median,maximum,IQR_lower_bound,IQR_upper_bound,IQR_outlier_count
0,products,price,2000,5.14,240.72,3492.50,-622.32,1268.25,163
1,products,compare_at_price,355,8.65,286.98,5049.64,-812.15,1643.19,31
2,products,weight_g,2000,87.00,5906.50,11993.00,-5723.12,17571.88,0
3,products,rating,1441,1.00,4.00,5.00,0.75,7.55,0
4,customers,orders_count,800,0.00,3.00,12.00,-2.50,9.50,5
5,customers,total_spent,800,0.00,3877.88,30063.98,-6554.02,14959.68,30
6,orders,subtotal,3000,5.14,744.48,16027.02,-1872.22,3837.65,207
7,orders,shipping,3000,0.00,5.99,29.99,-7.51,28.49,243
8,orders,tax,3000,0.36,51.05,1121.89,-129.73,264.40,209
9,orders,discount,3000,0.00,0.00,2421.08,0.00,0.00,420


In [8]:

products = processed["products"]

category_outliers = []

for category_id, group in products.groupby("category_id"):
    q1 = group["price"].quantile(0.25)
    q3 = group["price"].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (group["price"] < lower_bound)
        | (group["price"] > upper_bound)
    ).sum()

    category_outliers.append({
        "category_id": category_id,
        "product_count": len(group),
        "median_price": group["price"].median(),
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": int(outlier_count),
    })

category_outliers = pd.DataFrame(category_outliers)

display(category_outliers.round(2))


,category_id,product_count,median_price,lower_bound,upper_bound,outlier_count
0,1,244,433.86,-376.77,1256.91,0
1,2,260,1553.16,-1592.47,4985.00,0
2,3,260,179.08,-140.27,516.78,0
3,4,246,407.88,-350.45,1162.02,0
4,5,261,547.65,-772.05,1939.87,0
5,6,251,52.76,-22.30,124.48,0
6,7,225,99.46,-74.68,287.88,0
7,8,253,218.77,-221.64,657.96,0


### Outlier Investigation: Product Prices

The global IQR method identified 163 potential outliers among 2,000 product prices. However, applying the IQR method separately within each of the eight product categories identified no outliers.

This difference suggests that the global method may flag prices because product categories have substantially different price distributions, rather than because the observations are erroneous.

**Decision:**
- Preserve all original product prices.
- Do not remove or cap price outliers based on the global IQR method.
- Retain category information when analysing product prices.
- Reconsider transformations only if a specific downstream modelling task requires them.

An IQR outlier is a statistical flag, not proof of a data error.

In [9]:

missing_summary = []

for dataset_name, df in processed.items():
    for column in df.columns:
        missing_count = int(df[column].isna().sum())

        if missing_count > 0:
            missing_summary.append({
                "dataset": dataset_name,
                "column": column,
                "missing_count": missing_count,
                "missing_percentage": round(
                    missing_count / len(df) * 100, 2
                ),
                "dtype": str(df[column].dtype),
            })

missing_summary = pd.DataFrame(missing_summary)

display(
    missing_summary.sort_values(
        ["dataset", "missing_percentage"],
        ascending=[True, False],
    ).reset_index(drop=True)
)


,dataset,column,missing_count,missing_percentage,dtype
0,customers,notes,717,89.62,str
1,customers,company,563,70.38,str
2,orders,notes,2750,91.67,str
3,orders,delivered_at,325,10.83,"datetime64[us, UTC]"
4,orders,tracking_number,293,9.77,str
5,orders,shipped_at,293,9.77,"datetime64[us, UTC]"
6,orders,paid_at,7,0.23,"datetime64[us, UTC]"
7,products,compare_at_price,1645,82.25,float64
8,products,rating,559,27.95,float64


### Missing Values: Handling Strategy

Missing values were reviewed based on their meaning and potential analytical impact. No missing values are imputed at this stage.

| Dataset | Column | Missing % | Decision | Reason |
|---|---|---:|---|---|
| Customers | `notes` | 89.62% | Preserve as missing | Optional text; high missingness |
| Customers | `company` | 70.38% | Preserve as missing | May indicate an unavailable or unrecorded company |
| Orders | `notes` | 91.67% | Preserve as missing | Optional descriptive text |
| Orders | `delivered_at` | 10.83% | Preserve as missing | Delivery may not have occurred or may not have been recorded |
| Orders | `tracking_number` | 9.77% | Preserve as missing | Tracking information may be unavailable or inapplicable |
| Orders | `shipped_at` | 9.77% | Preserve as missing | Shipment may not have occurred or may not have been recorded |
| Orders | `paid_at` | 0.23% | Preserve as missing | Payment timestamp is unavailable for a small number of records |
| Products | `compare_at_price` | 82.25% | Preserve as missing | A reference/comparison price may not be provided |
| Products | `rating` | 27.95% | Preserve as missing | Previously verified to correspond to products without reviews |

**Preprocessing decision:** Preserve the missing values in the working data. Do not replace absent ratings with zero, missing timestamps with fabricated dates, or missing prices with assumed values.

For future modelling, any imputation or missingness indicators will be created only when justified by the task and fitted using training data where applicable.

The interpretation of missing order events remains provisional because the source system's status and timestamp semantics have not been independently verified.

In [10]:
orders = processed["orders"]

# Derive calendar features from the order placement timestamp
orders["placed_year"] = orders["placed_at"].dt.year
orders["placed_month"] = orders["placed_at"].dt.month
orders["placed_month_name"] = orders["placed_at"].dt.month_name()
orders["placed_day_of_week"] = orders["placed_at"].dt.day_name()
orders["placed_hour"] = orders["placed_at"].dt.hour
orders["placed_weekend"] = (
    orders["placed_at"].dt.dayofweek >= 5
)

# Validate the new features
display(
    orders[
        [
            "placed_at",
            "placed_year",
            "placed_month",
            "placed_month_name",
            "placed_day_of_week",
            "placed_hour",
            "placed_weekend",
        ]
    ].head(10)
)

print("Missing placed_at:", orders["placed_at"].isna().sum())
print("Missing placed_year:", orders["placed_year"].isna().sum())
print("Year range:", orders["placed_year"].min(), "-", orders["placed_year"].max())
print("Weekend orders:", orders["placed_weekend"].sum())

,placed_at,placed_year,placed_month,placed_month_name,placed_day_of_week,placed_hour,placed_weekend
0,2026-09-01 11:06:02+00:00,2026,9,September,Tuesday,11,False
1,2026-09-01 09:42:53+00:00,2026,9,September,Tuesday,9,False
2,2026-09-01 09:04:56+00:00,2026,9,September,Tuesday,9,False
3,2026-09-01 01:39:07+00:00,2026,9,September,Tuesday,1,False
4,2026-09-01 00:41:24+00:00,2026,9,September,Tuesday,0,False
5,2026-08-31 16:13:54+00:00,2026,8,August,Monday,16,False
6,2026-08-31 15:23:09+00:00,2026,8,August,Monday,15,False
7,2026-08-31 14:16:42+00:00,2026,8,August,Monday,14,False
8,2026-08-31 06:13:58+00:00,2026,8,August,Monday,6,False
9,2026-08-31 05:57:38+00:00,2026,8,August,Monday,5,False


Missing placed_at: 0
Missing placed_year: 0
Year range: 2025 - 2026
Weekend orders: 878


In [11]:

orders = processed["orders"]

# Calculate elapsed time in hours when both timestamps exist
orders["hours_to_payment"] = (
    orders["paid_at"] - orders["placed_at"]
).dt.total_seconds() / 3600

orders["hours_to_shipment"] = (
    orders["shipped_at"] - orders["paid_at"]
).dt.total_seconds() / 3600

orders["hours_to_delivery"] = (
    orders["delivered_at"] - orders["shipped_at"]
).dt.total_seconds() / 3600

duration_columns = [
    "hours_to_payment",
    "hours_to_shipment",
    "hours_to_delivery",
]

# Summarise coverage and invalid negative durations
for column in duration_columns:
    values = orders[column]

    print(f"\n{column}")
    print("Available durations:", values.notna().sum())
    print("Missing durations:", values.isna().sum())
    print("Negative durations:", (values < 0).sum())

    print(
        values.describe()[
            ["mean", "50%", "min", "max"]
        ].round(2)
    )



hours_to_payment
Available durations: 2993
Missing durations: 7
Negative durations: 0
mean    1.02
50%     1.02
min     0.02
max     2.00
Name: hours_to_payment, dtype: float64

hours_to_shipment
Available durations: 2707
Missing durations: 293
Negative durations: 0
mean    25.83
50%     25.52
min      4.00
max     47.97
Name: hours_to_shipment, dtype: float64

hours_to_delivery
Available durations: 2675
Missing durations: 325
Negative durations: 0
mean     84.66
50%      84.62
min      24.00
max     143.93
Name: hours_to_delivery, dtype: float64


### Order Processing Time Features

Three elapsed-time features were derived from the available order timestamps:

- `hours_to_payment`: placement to payment.
- `hours_to_shipment`: payment to shipment.
- `hours_to_delivery`: shipment to delivery.

No negative durations were found.

| Feature | Median duration | Available records |
|---|---:|---:|
| `hours_to_payment` | 1.02 hours | 2,993 |
| `hours_to_shipment` | 25.52 hours | 2,707 |
| `hours_to_delivery` | 84.62 hours | 2,675 |

Missing durations were retained when one or both required timestamps were unavailable. These statistics describe the supplied dataset and should not be treated as industry benchmarks. The underlying timestamp-generation and recording processes have not been independently verified.

In [12]:

customers = processed["customers"]
orders = processed["orders"]

# Aggregate all recorded orders per customer.
# These include every status, so totals are not necessarily realized revenue.
customer_order_aggregates = (
    orders.groupby("customer_id")
    .agg(
        calculated_orders_count=("id", "nunique"),
        calculated_order_total=("total", "sum"),
        recorded_order_rows=("id", "size"),
    )
    .reset_index()
    .rename(columns={"customer_id": "id"})
)

# Merge aggregates into a separate customer-level working table
customer_features = customers.merge(
    customer_order_aggregates,
    on="id",
    how="left",
    validate="one_to_one",
)

# Customers without recorded orders receive zero for these derived counts/totals
customer_features["calculated_orders_count"] = (
    customer_features["calculated_orders_count"].fillna(0).astype(int)
)

customer_features["calculated_order_total"] = (
    customer_features["calculated_order_total"].fillna(0)
)

customer_features["recorded_order_rows"] = (
    customer_features["recorded_order_rows"].fillna(0).astype(int)
)

# Compare source metrics with aggregates calculated from all order statuses
customer_features["orders_count_matches"] = (
    customer_features["orders_count"]
    == customer_features["calculated_orders_count"]
)

customer_features["total_spent_matches"] = np.isclose(
    customer_features["total_spent"],
    customer_features["calculated_order_total"],
    atol=0.01,
)

print(
    "orders_count matches:",
    customer_features["orders_count_matches"].sum(),
    "/",
    len(customer_features),
)

print(
    "total_spent matches:",
    customer_features["total_spent_matches"].sum(),
    "/",
    len(customer_features),
)

display(
    customer_features[
        [
            "id",
            "orders_count",
            "calculated_orders_count",
            "total_spent",
            "calculated_order_total",
            "orders_count_matches",
            "total_spent_matches",
        ]
    ].head(10)
)


orders_count matches: 566 / 800
total_spent matches: 566 / 800


,id,orders_count,calculated_orders_count,total_spent,calculated_order_total,orders_count_matches,total_spent_matches
0,1,4,6,5419.75,8163.84,False,False
1,2,4,4,11795.61,11795.61,True,True
2,3,5,5,3138.05,3138.05,True,True
3,4,2,2,468.40,468.40,True,True
4,5,3,4,3787.59,6204.91,False,False
5,6,0,1,0.00,308.56,False,False
6,7,4,4,4967.00,4967.00,True,True
7,8,2,3,1765.24,2091.88,False,False
8,9,2,2,612.14,612.14,True,True
9,10,4,5,5882.65,6678.86,False,False


### Customer-Level Metric Validation

Customer metrics were compared with aggregates recalculated from all recorded orders, including every order status.

- `orders_count` matched for 566 of 800 customers (70.75%).
- `total_spent` matched for 566 of 800 customers (70.75%).
- Each metric had 234 mismatches.

For example, Customer 1 has 4 orders and $5,419.75 in the customer table, compared with 6 orders and $8,163.84 calculated from the orders table. Customer 6 has zero recorded orders in the customer metrics but one order in the orders table.

**Decision:** Preserve the original customer metrics. Keep the recalculated aggregates as separate derived features, and do not assume that all order statuses represent completed purchases or realized revenue. The business definitions and refresh timing of the source metrics remain unverified.

In [13]:
# Validate engineered features without modifying source columns

print("ORDER FEATURE VALIDATION")
print("Rows:", len(processed["orders"]))
print("Unique order IDs:", processed["orders"]["id"].nunique())

for column in [
    "placed_year",
    "placed_month",
    "placed_hour",
    "placed_weekend",
    "hours_to_payment",
    "hours_to_shipment",
    "hours_to_delivery",
]:
    print(
        f"{column}: "
        f"missing={processed['orders'][column].isna().sum()}, "
        f"dtype={processed['orders'][column].dtype}"
    )

print("\nCUSTOMER FEATURE VALIDATION")
print("Rows:", len(customer_features))
print("Unique customer IDs:", customer_features["id"].nunique())
print(
    "Customers without recorded orders:",
    (customer_features["calculated_orders_count"] == 0).sum()
)
print(
    "Negative calculated order totals:",
    (customer_features["calculated_order_total"] < 0).sum()
)

print("\nRAW DATA PRESERVATION")
for name in data:
    print(
        f"{name}:",
        "same row count =", len(data[name]) == len(processed[name]),
        "| raw data unchanged =", data[name].equals(
            pd.read_csv(RAW_DATA_DIR / f"{name}.csv")
        )
    )

ORDER FEATURE VALIDATION
Rows: 3000
Unique order IDs: 3000
placed_year: missing=0, dtype=int32
placed_month: missing=0, dtype=int32
placed_hour: missing=0, dtype=int32
placed_weekend: missing=0, dtype=bool
hours_to_payment: missing=7, dtype=float64
hours_to_shipment: missing=293, dtype=float64
hours_to_delivery: missing=325, dtype=float64

CUSTOMER FEATURE VALIDATION
Rows: 800
Unique customer IDs: 800
Customers without recorded orders: 23
Negative calculated order totals: 0

RAW DATA PRESERVATION
products: same row count = True | raw data unchanged = True
customers: same row count = True | raw data unchanged = True
orders: same row count = True | raw data unchanged = True
order_items: same row count = True | raw data unchanged = True
reviews: same row count = True | raw data unchanged = True


In [14]:
# Candidate features for a future modelling pipeline
# This is an inventory only; no encoding or scaling is applied.

model_feature_candidates = {
    "products": {
        "categorical": [
            "category_id",
            "brand",
            "color",
            "status",
            "in_stock",
        ],
        "numeric": [
            "price",
            "weight_g",
            "review_count",
        ],
    },
    "customers": {
        "categorical": [
            "tier",
            "marketing_opt_in",
        ],
        "numeric": [
            "orders_count",
            "total_spent",
        ],
    },
    "orders": {
        "categorical": [
            "status",
            "payment_method",
            "shipping_method",
            "placed_weekend",
        ],
        "numeric": [
            "subtotal",
            "shipping",
            "tax",
            "discount",
            "total",
            "placed_month",
            "placed_hour",
            "hours_to_payment",
            "hours_to_shipment",
            "hours_to_delivery",
        ],
    },
}

for dataset_name, feature_groups in model_feature_candidates.items():
    df = processed[dataset_name]

    print(f"\n{dataset_name.upper()}")

    for group_name, columns in feature_groups.items():
        available = [column for column in columns if column in df.columns]
        print(f"{group_name}: {available}")

        for column in available:
            print(
                f"  {column}: dtype={df[column].dtype}, "
                f"missing={df[column].isna().sum()}, "
                f"unique={df[column].nunique(dropna=True)}"
            )


PRODUCTS
categorical: ['category_id', 'brand', 'color', 'status', 'in_stock']
  category_id: dtype=int64, missing=0, unique=8
  brand: dtype=str, missing=0, unique=32
  color: dtype=str, missing=0, unique=10
  status: dtype=str, missing=0, unique=3
  in_stock: dtype=bool, missing=0, unique=2
numeric: ['price', 'weight_g', 'review_count']
  price: dtype=float64, missing=0, unique=1975
  weight_g: dtype=int64, missing=0, unique=1844
  review_count: dtype=int64, missing=0, unique=7

CUSTOMERS
categorical: ['tier', 'marketing_opt_in']
  tier: dtype=str, missing=0, unique=4
  marketing_opt_in: dtype=bool, missing=0, unique=2
numeric: ['orders_count', 'total_spent']
  orders_count: dtype=int64, missing=0, unique=13
  total_spent: dtype=float64, missing=0, unique=768

ORDERS
categorical: ['status', 'payment_method', 'shipping_method', 'placed_weekend']
  status: dtype=str, missing=0, unique=6
  payment_method: dtype=str, missing=0, unique=4
  shipping_method: dtype=str, missing=0, unique=4
 

## Preprocessing Decisions and Validation Summary

### 1. Data integrity
- Created separate working copies of all five datasets.
- Preserved the original raw CSV files.
- Confirmed that row counts remain unchanged and IDs remain unique in the validated order and customer tables.
- Converted timestamp columns to UTC-aware datetime values without introducing additional missing timestamps.

### 2. Missing values
- Preserved optional text fields and unavailable product comparison prices as missing.
- Preserved missing product ratings rather than assigning zero to products without reviews.
- Preserved missing order timestamps and tracking numbers.
- Retained missing processing durations when their required timestamps were unavailable.

### 3. Outliers
- Identified potential outliers using the IQR method.
- Found 163 global product-price outliers, but no category-specific price outliers across the eight categories.
- Did not delete or cap observations solely because they were flagged by IQR.

### 4. Feature engineering
- Created calendar features from `placed_at`, including year, month, month name, weekday, hour, and weekend indicator.
- Derived elapsed-time features for payment, shipment, and delivery stages.
- Recalculated customer order counts and recorded order totals in separate derived columns.
- Preserved original customer metrics because their definitions did not consistently match aggregates from all order statuses.

### 5. Encoding and scaling
- Identified candidate categorical and numerical features for future modelling.
- Did not apply global encoding or scaling because these operations should be fitted within a model-specific pipeline, using training data only.
- Target definition and leakage risks must be reviewed before selecting final model features.

### Conclusion
The preprocessing investigation produced validated derived features and documented handling decisions while preserving the original source data. No rows were removed, and no missing values or outliers were imputed, deleted, or capped during this stage.

In [15]:
from pathlib import Path

# Create output directory
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Save processed datasets as CSV files
for dataset_name, df in processed.items():
    output_path = PROCESSED_DATA_DIR / f"{dataset_name}_processed.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path.name} | Shape: {df.shape}")

# Save customer-level derived features separately
customer_features_path = (
    PROCESSED_DATA_DIR / "customer_features.csv"
)

customer_features.to_csv(customer_features_path, index=False)

print(f"\nSaved: {customer_features_path.name}")
print(f"Output directory: {PROCESSED_DATA_DIR}")

Saved: products_processed.csv | Shape: (2000, 19)
Saved: customers_processed.csv | Shape: (800, 13)
Saved: orders_processed.csv | Shape: (3000, 30)
Saved: order_items_processed.csv | Shape: (6219, 10)
Saved: reviews_processed.csv | Shape: (2500, 10)

Saved: customer_features.csv
Output directory: c:\Users\Lenovo\ecommerce-customer-behaviour\data\processed


In [16]:
# Reload all processed datasets from disk
reloaded = {
    name: pd.read_csv(
        PROCESSED_DATA_DIR / f"{name}_processed.csv"
    )
    for name in data
}

# Validate row counts and columns
for name, df in reloaded.items():
    expected = processed[name]
    reloaded_df = df

    print(f"\n{name.upper()}")
    print("Shape:", reloaded_df.shape)
    print("Expected shape:", expected.shape)
    print("Columns preserved:", list(reloaded_df.columns) == list(expected.columns))
    print("Row count preserved:", len(reloaded_df) == len(expected))

# Validate separately saved customer features
reloaded_customer_features = pd.read_csv(
    PROCESSED_DATA_DIR / "customer_features.csv"
)

print("\nCUSTOMER FEATURES")
print("Shape:", reloaded_customer_features.shape)
print("Unique customer IDs:", reloaded_customer_features["id"].nunique())

# Confirm raw files are untouched
print("\nRAW FILES")
for name, original_df in data.items():
    raw_disk = pd.read_csv(RAW_DATA_DIR / f"{name}.csv")
    print(f"{name}: unchanged =", original_df.equals(raw_disk))


PRODUCTS
Shape: (2000, 19)
Expected shape: (2000, 19)
Columns preserved: True
Row count preserved: True

CUSTOMERS
Shape: (800, 13)
Expected shape: (800, 13)
Columns preserved: True
Row count preserved: True

ORDERS
Shape: (3000, 30)
Expected shape: (3000, 30)
Columns preserved: True
Row count preserved: True

ORDER_ITEMS
Shape: (6219, 10)
Expected shape: (6219, 10)
Columns preserved: True
Row count preserved: True

REVIEWS
Shape: (2500, 10)
Expected shape: (2500, 10)
Columns preserved: True
Row count preserved: True

CUSTOMER FEATURES
Shape: (800, 18)
Unique customer IDs: 800

RAW FILES
products: unchanged = True
customers: unchanged = True
orders: unchanged = True
order_items: unchanged = True
reviews: unchanged = True


## Final Conclusion

The heavy preprocessing investigation is complete.

The five datasets were loaded into separate working copies, timestamp columns were converted to UTC-aware datetime values, missingness and outliers were investigated, and derived temporal and customer-level features were created.

Key decisions:
- No rows were removed.
- Missing values were preserved rather than imputed without justification.
- Potential outliers were investigated but not deleted or capped.
- Original customer metrics were preserved despite inconsistencies with recalculated order aggregates.
- Derived datasets were saved under `data/processed/` and successfully reloaded with their expected shapes and columns.
- The original files under `data/raw/` remained unchanged.

The processed datasets are ready for further exploratory and statistical investigation. Any future machine-learning preprocessing must be designed around a clearly defined prediction target and fitted without data leakage.